### Regressiooni otsustusmets rohtse biomassi ennustamiseks Lääne-Eesti loopealsetel, suvi 2019
Sentinel-1 ja Sentinel-2 mediaanväärtused 2019 suvest (juuni - august)



Iris Luik, 2026

Tartu Ülikool, Geograafia osakond

In [ ]:
import os
# kirjutan üle
os.environ['PROJ_DATA'] = r'C:\Users\irisl\micromamba\envs\geopython2025\Library\share\proj'
os.environ['PROJ_LIB'] = r'C:\Users\irisl\micromamba\envs\geopython2025\Library\share\proj'

In [ ]:
# vajaminevad paketid

import random
import glob

# data
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.stats import shapiro

# visualiseerimine
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns
import contextily as ctx

# masinõpe
from sklearn.model_selection import cross_validate, train_test_split, KFold, cross_val_score, cross_val_predict
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.inspection import PartialDependenceDisplay
import shap
from joblib import dump, load

# raster
import rasterio
from rasterio.features import rasterize
from affine import Affine

In [ ]:
# kaugseire andmed csv failidest

# punktipõhised andmed
fp = "C:/Users/irisl/geopython2025/BIOMASS/gee_data/mudel/gee_results_points_2019.csv"
obs_data = pd.read_csv(fp)

# 3x3 kerneli meetodiga andmed
fp_kernel = "C:/Users/irisl/geopython2025/BIOMASS/gee_data/mudel/gee_results_points_2019_kernel3x3.csv"
obs_data_kernel = pd.read_csv(fp_kernel)

obs_data.columns

In [ ]:
# kontrollin kas ID-d on unikaalsed

obs_data['ID'].is_unique
#obs_data_kernel['ID'].is_unique

In [ ]:
# loen välitööde andmetega csv faili sisse

fp = "C:/Users/irisl/geopython2025/BIOMASS/KIK_data/KIK2019.csv"
kik_data = pd.read_csv(fp)
kik_data.columns

In [ ]:
# liidan välitööde andmed kaugseire andmetega proovialade ID alusel

data = obs_data.merge(kik_data, on="ID", how="left", suffixes=('', '_kik')) # punktipõhine
data_kernel = obs_data_kernel.merge(kik_data, on="ID", how="left", suffixes=('', '_kik')) #kerneli meetod

In [ ]:
# ridade arv

len(data)

In [ ]:
data.head(5)

In [ ]:
# teisendan biomassi t/ha -> g/m2 
# rohtne biomass on Biomass_womoss


data["Biomass_womoss_gm2"] = data["Biomass_womoss_tha"] * 100
data["Biomass_gm2"] = data["Biomass_tha"] * 100

data_kernel["Biomass_womoss_gm2"] = data_kernel["Biomass_womoss_tha"] * 100
data_kernel["Biomass_gm2"] = data_kernel["Biomass_tha"] * 100

In [ ]:
# rohtse biomassi histogramm

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    data["Biomass_womoss_gm2"],
    bins=30,
    color="#51915C",
    edgecolor="black",
    alpha=0.8
)

ax.set_xlabel("Rohtne biomass (g/m²)")
ax.set_ylabel("Vaatluste arv")

plt.tight_layout()
plt.show()

In [ ]:
# rohtse biomassi histogramm ala iseloomu (Subsite) alusel

# vahemikud
min_biomass = data["Biomass_womoss_gm2"].min()
max_biomass = data["Biomass_womoss_gm2"].max()
bins = np.linspace(min_biomass, max_biomass, 41)


#  tõlge
subsite_names_et = {
    "Open": "Avatud",
    "Overgrown": "Ülekasvanud",
    "Afforested": "Metsastunud",
    "kontroll": "Kontroll"
}

subsites = ["Open", "Overgrown", "Afforested", "kontroll"]

fig, axs = plt.subplots(1, 4, figsize=(20, 5), sharey=True, sharex=True)

for ax, subsite in zip(axs, subsites):
    subset = data[data["Subsite"] == subsite]["Biomass_womoss_gm2"]

    ax.hist(
        subset,
        bins=bins,
        color="#51915C",
        edgecolor="black",
        alpha=0.8
    )

    ax.set_title(f"{subsite_names_et[subsite]}\n(n = {len(subset)})")
    ax.set_xlabel("Rohtne biomass (g/m²)")

axs[0].set_ylabel("Vaatluste arv")

fig.suptitle("Rohtse biomassi jaotus prooviala iseloomu alusel", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# eemaldan metsastunud alad - punktipõhine
cleaned = data[data["Subsite"] != "Afforested"]

# eemaldan metsastunud alad - kernel
cleaned_kernel = data_kernel[data_kernel["Subsite"] != "Afforested"]

# Kontrollin metsata punktide arvu
print("Metsata vaatlusalade arv punktipõhise meetodi puhul:", len(cleaned))
print("Metsata vaatlusalade arv kerneli puhul:", len(cleaned_kernel))

In [ ]:
# mudelis kasutatud alade bioamassi histogramm töösse

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    cleaned["Biomass_womoss_gm2"],
    bins=30,
    color="#51915C",
    edgecolor="black",
    alpha=0.8
)

ax.set_xlabel("Rohtne biomass (g/m²)")
ax.set_ylabel("Vaatluste arv")

# y-telje ainult täisarvud
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

In [ ]:
# mitu vaatlust on üle 500 g/m²
num_over_500 = (cleaned["Biomass_womoss_gm2"] > 500 ).sum()
print(f"Punkte üle 500 g/m²: {num_over_500}")

In [ ]:
# metsata proovialade keskmine rohtne biomass g/m2
print("Metsata proovialade keskmine rohtne biomass: ", round(data['Biomass_womoss_gm2'].mean(),1), "g/m2")

In [ ]:
# metsata proovialade mediaan rohtne biomass g/m2
print("Metsata proovialade mediaan rohtne biomass: ", round(data['Biomass_womoss_gm2'].median(),1), "g/m2")

In [ ]:
# kontrollin normaaljaotust Shapiro testiga

stat, p = shapiro(cleaned["Biomass_gm2"])
print("p-value:", p)

## Tunnuste valik

In [ ]:
# Tunnuste valikul lähtuti Spearmani korrelatsioonikoefitsendist - punktipõhine

features = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12", "VH", "VV", "VV_VH", "VH_VV", "BSI", "NDVI", "SAVI", "X", "Y"]

corr = (
    cleaned[features + ["Biomass_womoss_gm2"]]
    .corr(method="spearman")["Biomass_womoss_gm2"]
    .drop("Biomass_womoss_gm2")
    .sort_values(ascending=False)
)

corr

# valin ainult need tunnused, mille korrelatsioon sihtmuutujaga > |-0,4| (S2 mudel)
# valin ainult need tunnused, mille korrelatsioon sihtmuutujaga > |-0,3| (S1, S2 mudel)

In [ ]:
# Spearmani korrelatsioonikoefitsent - kernel

features_kernel = ["B2_mean", "B3_mean", "B4_mean", "B5_mean", "B6_mean", "B7_mean", "B8_mean", "B8A_mean", "B11_mean", "B12_mean", "VH_mean", "VV_mean", "VH_VV_mean", "BSI_mean", "NDVI_mean", "SAVI_mean", "X", "Y"] # kernel


corr_kernel = (
    cleaned_kernel[features_kernel + ["Biomass_womoss_gm2"]]
    .corr(method="spearman")["Biomass_womoss_gm2"]
    .drop("Biomass_womoss_gm2")
    .sort_values(ascending=False)
)

corr_kernel

## Masinõppe mudel

In [ ]:
#S1 ja S2 KOMBOMUDEL

X_train = cleaned[["SAVI", "NDVI", "B7", "B8", "B8A", "B6", "VH", "VH_VV", "B4", "B11", "B12", "BSI"]] #S1 ja S2 KOMBOMUDEL
#X_train = cleaned_kernel[["SAVI_mean", "NDVI_mean", "B7_mean", "B8_mean", "B8A_mean", "B6_mean", "VH_mean", "VH_VV_mean", "B4_mean", "B11_mean", "B12_mean", "BSI_mean"]] #S1 ja S2 KOMBOMUDEL kerneliga


#S2 MUDEL

#X_train = cleaned[["SAVI", "NDVI", "B7", "B8", "B8A", "B6", "B12", "BSI"]] #S2mudel
#X_train = cleaned_kernel[["SAVI_mean", "NDVI_mean", "B7_mean", "B8_mean", "B8A_mean", "B6_mean", "B12_mean", "BSI_mean"]] #S2mudel kerneliga


y_train = cleaned["Biomass_womoss_gm2"]
#y_train = cleaned_kernel["Biomass_womoss_gm2"]

random_state = 1 # juhuarv

results = []

# 5-kordne ristvalideerimine
cv = KFold(n_splits=5, shuffle=True, random_state=random_state)

# r2, RMSE ja MAE
scoring = {
    "r2": "r2",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error"
}

for est in [50, 75, 100, 300, 500]:
    for depth in [3, 5, 7, 10]:
        for leaf in [3, 5, 10]:

            rf = RandomForestRegressor(
                n_estimators=est,
                max_depth=depth,
                min_samples_leaf=leaf,
                max_features='sqrt',
                random_state=random_state
            )

            cv_results = cross_validate(
                rf,
                X_train,
                y_train,
                cv=cv,
                scoring=scoring,
                n_jobs=-1
            )

            results.append({
                "n_estimators": est,
                "max_depth": depth,
                "min_samples_leaf": leaf,
                "mean_r2": cv_results["test_r2"].mean(),
                "std_r2": cv_results["test_r2"].std(),
                "mean_rmse": -cv_results["test_rmse"].mean(),
                "std_rmse": cv_results["test_rmse"].std(),
                "mean_mae": -cv_results["test_mae"].mean(),
                "std_mae": cv_results["test_mae"].std()
            })

results_df = pd.DataFrame(results)


# sorteerin tulemused keskmise rmse alusel
results_df = results_df.sort_values("mean_rmse", ascending=True)

results_df.head(10)

In [ ]:
# valin parimad hüperparameetrid eelmise tulemuse alusel

best = results_df.iloc[0]
best_params = {
    "n_estimators": int(best["n_estimators"]),
    "max_depth": int(best["max_depth"]),
    "min_samples_leaf": int(best["min_samples_leaf"]),
}

print("Parimad hüperparameetrid:")
print(best_params)

In [ ]:
regressor = RandomForestRegressor(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    min_samples_leaf=best_params["min_samples_leaf"],
    max_features='sqrt',
    random_state=random_state)

# mudel
regressor.fit(X_train, y_train)


# ennustused
cv = KFold(n_splits=5, shuffle=True, random_state=random_state)
y_pred_cv = cross_val_predict(regressor, X_train, y_train, cv=cv, n_jobs=-1)

# ennustustäpsuse näitajad
r2_mean, r2_std     = best["mean_r2"], best["std_r2"]
rmse_mean, rmse_std = best["mean_rmse"], best["std_rmse"]
mae_mean, mae_std   = best["mean_mae"], best["std_mae"]


y_true = y_train.values
minv = min(y_true.min(), y_pred_cv.min())
maxv = max(y_true.max(), y_pred_cv.max()) + 50

# joonis
plt.figure(figsize=(6.5, 6.5))
plt.scatter(y_true, y_pred_cv, alpha=0.7, color="#51915C")
plt.plot([minv, maxv], [minv, maxv], linestyle="--", color="#2F5D37")

plt.xlabel("Tegelik rohtne biomass (g/m²)")
plt.ylabel("Ennustatud rohtne biomass (g/m²)")

plt.text(
    0.05, 0.95,
    f"$R^2$ = {r2_mean:.2f} ± {r2_std:.2f}\n"
    f"RMSE = {round(rmse_mean)} ± {round(rmse_std)} g/m²\n"
    f"MAE = {round(mae_mean)} ± {round(mae_std)} g/m²",
    transform=plt.gca().transAxes,
    va="top"
)

plt.xlim(minv, maxv)
plt.ylim(minv, maxv)
plt.tight_layout()
plt.savefig("scatter_plot.png", dpi=300) # save
plt.show()

In [ ]:
# jäägid

resid = y_true - y_pred_cv

plt.figure(figsize=(6.5, 5))
plt.axhline(0, linestyle="--", color="#2F5D37")
plt.scatter(y_pred_cv, resid, alpha=0.7, color="#51915C")


plt.xlabel("Ennustatud rohtne biomass (g/m²)")
plt.ylabel("Jääk (tegelik - ennustatud) (g/m²)")
plt.tight_layout()
plt.savefig("residuals_vs_pred.png", dpi=300)
plt.show()

In [ ]:
# arvutan SHAP väärtused

explainer = shap.TreeExplainer(regressor)
shap_values = explainer.shap_values(X_train)

# nimetan ümber

name_shap = {"VH_VV": "VH/VV"}
X_plot = X_train.rename(columns=name_shap)

In [ ]:
# SHAP joonis

plt.figure(figsize=(12, 8))

shap.summary_plot(shap_values=shap_values, 
                  features=X_plot, 
                  feature_names=X_plot.columns,
                  show=False)  

plt.tight_layout() 
plt.savefig("shap_summary_plot.png", dpi=300)  # 300 dpi töös kasutamiseks
plt.show()

In [ ]:
# osalise sõltuvuse graafik (partial dependence plot)

fig, ax = plt.subplots(figsize=(12, 12))
disp = PartialDependenceDisplay.from_estimator(regressor, X_train, X_train.columns, ax=ax)
fig.subplots_adjust(hspace=0.3)
fig.tight_layout()

In [ ]:
# lisan ennustused ja jäägid vaatlusandmetele

cleaned_pred = cleaned.copy()
cleaned_pred["pred_biomass_womoss_gm2"] = regressor.predict(X_train)

cleaned_pred["residual"] = cleaned_pred["Biomass_womoss_gm2"] - cleaned_pred["pred_biomass_womoss_gm2"]
cleaned_pred["abs_residual"] = cleaned_pred["residual"].abs() # jääkide absoluutväärtus
cleaned_pred.head(5)

In [ ]:
# teen geodataframe'i visualiseerimiseks

cleaned_pred_gdf = gpd.GeoDataFrame(
    cleaned_pred,
    geometry=gpd.points_from_xy(cleaned_pred.X, cleaned_pred.Y),
    crs="EPSG:4326"
)

In [ ]:
# interaktiivne kaart ennustatud väärtustega

cleaned_pred_gdf.explore(
    column="pred_biomass_womoss_gm2",
    cmap="YlGn",
    tooltip=["ID", "Biomass_womoss_gm2", "pred_biomass_womoss_gm2"],
    marker_kwds={"radius": 4},
    style_kwds={"color": "black", "weight": 1, "fillOpacity": 0.9}
)

In [ ]:
# interaktiivne kaart jääkide jaoks

cleaned_pred_gdf.explore(
    column="residual",
    cmap="RdBu_r",
    tooltip=["ID", "Biomass_womoss_gm2", "pred_biomass_womoss_gm2", "residual"],
    marker_kwds={"radius": 4},
    style_kwds={"color": "black", "weight": 1, "fillOpacity": 0.9},
    #vmin=-5,  # Set the minimum color value
    #vmax=5   # Set the maximum color value
)

In [ ]:
# salvestan geopakina

cleaned_pred_gdf.to_file("biomass_predictions.gpkg", layer="biomass_predictions", driver="GPKG")

In [ ]:
# ennustuste histogramm

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    cleaned_pred["pred_biomass_womoss_gm2"],
    bins=20,
    color="#51915C",
    edgecolor="black",
    alpha=0.8
)

ax.set_xlabel("Ennustatud rohtne biomass (g/m²)")
ax.set_ylabel("Vaatluste arv")

plt.tight_layout()
plt.show()

In [ ]:
# keskmine ennustus

cleaned_pred["pred_biomass_womoss_gm2"].mean()

In [ ]:
# mediaan ennustus

cleaned_pred["pred_biomass_womoss_gm2"].median()

In [ ]:
# jääkide histogramm

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    cleaned_pred["residual"],
    bins=30,
    color="#51915C",
    edgecolor="black",
    alpha=0.8
)

ax.set_xlabel("Jääk (g/m²)")
ax.set_ylabel("Vaatluste arv")

plt.tight_layout()
plt.show()

In [ ]:
# keskmine jääk

cleaned_pred["residual"].abs().mean()

In [ ]:
# mediaan jääk

cleaned_pred["residual"].abs().median()

In [ ]:
((cleaned_pred["residual"] >= -50) & (cleaned_pred["residual"] <= 50)).sum()

### Ennustus 10 m punktivõrele

In [ ]:
# GEE csv failide kaust
path = "C:/Users/irisl/geopython2025/BIOMASS/gee_data/modelleerimine/*.csv"

# leian kõik csv failid
files = glob.glob(path)

# loen ja liidan üheks dataframeks
grid_gee_data = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

grid_gee_data.head()

In [ ]:
# kontrollin puuduvaid väärtuseid
grid_gee_data.isna().sum()

In [ ]:
# kontrollin punktide arvu (peab olema 645990)
len(grid_gee_data)

In [ ]:
# kontrollin veerge
grid_gee_data.columns

In [ ]:
# valin kombomudeli sisendtunnused
X_grid = grid_gee_data[["SAVI", "NDVI", "B7", "B8", "B8A", "B6", "VH", "VH_VV", "B4", "B11", "B12", "BSI"]]

In [ ]:
# ennustan kogu punktivõrele

predictions = regressor.predict(X_grid)
grid_prediction = grid_gee_data.loc[X_grid.index].copy()
grid_prediction['predicted_biomass_gm2'] = predictions

# punktivõre keskmine rohtne biomass

grid_prediction['predicted_biomass_gm2'].mean()

In [ ]:
# punktivõre mediaan rohtne biomass

grid_prediction['predicted_biomass_gm2'].median()

In [ ]:
# ennustuste histogramm

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    grid_prediction["predicted_biomass_gm2"],
    bins=70,
    color="#51915C",
    edgecolor="black",
    alpha=0.8
)

ax.set_xlabel("Ennustatud rohtne biomass (g/m²)")
ax.set_ylabel("Vaatluste arv")

plt.tight_layout()
plt.show()

In [ ]:
# teen geodataframe visualiseerimiseks
grid_gee_data_gdf = gpd.GeoDataFrame(grid_prediction, geometry=gpd.points_from_xy(grid_prediction["longitude"], grid_prediction["latitude"]), crs="EPSG:4326")

# teisendan Eesti koordinaatsüsteemi
grid_gee_data_gdf = grid_gee_data_gdf.to_crs("EPSG:3301")

# visualiseerin kontrollimiseks

fig, ax = plt.subplots(figsize=(12, 12))

grid_gee_data_gdf.plot(column='predicted_biomass_gm2', ax=ax, legend=True, cmap='YlGn', markersize=0.5, legend_kwds={'label': 'Rohtne biomass (g/m²)'})

# Add basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronNoLabels, crs='EPSG:3301')

plt.show()

### Tulemuse rastriks tegemine

In [ ]:
# esimese sammuna teisendan biomassi täisarvuks

grid_gee_data_gdf['pred_biomass_gm2_int'] = grid_gee_data_gdf['predicted_biomass_gm2'].round().astype('Int32')
grid_gee_data_gdf.head(5)

In [ ]:
# tekitan punktidest piksli kujud

xmin, ymin, xmax, ymax = grid_gee_data_gdf.total_bounds # ulatus

pixel_size = 10  # 10 m võre

width = int((xmax - xmin) / pixel_size)
height = int((ymax - ymin) / pixel_size)

transform = Affine(pixel_size, 0, xmin - pixel_size/2, 0, -pixel_size, ymax + pixel_size/2) # pikslite keskpunkid võre punktidel

shapes = ((geom, value)
    for geom, value in zip(
        grid_gee_data_gdf.geometry,
        grid_gee_data_gdf['pred_biomass_gm2_int']
    )
)

In [ ]:
# rasteriseerimine

raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=-9999,   # nodata väärtus
    dtype='int16' # int16, et mahtu kokku hoida
)

In [ ]:
# kirjutan geotif failiks

with rasterio.open(
    "biomass_prediction_alvars_10m.tif",
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype='int16',
    crs=grid_gee_data_gdf.crs,
    transform=transform,
    nodata=-9999
) as dst:
    dst.write(raster, 1)

In [ ]:
# avan rastri uuesti
with rasterio.open('biomass_prediction_alvars_10m.tif') as src:
    biomass = src.read(1)
    bounds = src.bounds
    transform = src.transform
    crs = src.crs